# Preconditions
`./setup_auth.ipynb` and `./setup_catalog_policies.ipynb`will run

In [1]:
%run ./setup_catalog_policies.ipynb

/Users/apabook/Desktop/ds-protocol/static/tutorial/venv/bin/python
{
    "participant_id": "did:jwk:provider",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2026-03-11T16:43:26.210970",
    "last_interaction": "2026-03-12T14:43:35.849845",
    "is_me": true
}
{
    "participant_id": "did:jwk:consumer",
    "participant_slug": "Consumer",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1100",
    "token": null,
    "saved_at": "2026-03-11T16:43:26.205695",
    "last_interaction": "2026-03-12T14:43:35.107610",
    "is_me": true
}
Provider DID: did:jwk:provider

Provider token: token

Consumer DID: did:jwk:consumer

Consumer token: token
{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:11c37141-dba8-4c7a-83fa-5a26a08e6eac",
    "dctIssued": "2026-03-11T16:43:26.223934Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCata

# Contract Negotiation

## Initialization of negotiation request (Consumer -> Provider)

In [2]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:d616721a-e9d2-419e-9d72-a051986bd534",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8d6030c4-2d14-4751-9815-d863e4031fba"
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f",
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:b055

## Provider creates initial offer (Provider -> Consumer)

In [3]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:d616721a-e9d2-419e-9d72-a051986bd534",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8d6030c4-2d14-4751-9815-d863e4031fba"
    },
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f",
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:ea1d0942-5ebe-4272-80

## Consumer sends negotiation request based on offer (Consumer -> Provider)

In [4]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:d616721a-e9d2-419e-9d72-a051986bd534",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8d6030c4-2d14-4751-9815-d863e4031fba"
    },
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f",
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:b0553918-b46a-4690-872f-e69cc7fd75fd",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http

## Provider updates/confirms the offer (Provider -> Consumer)

In [5]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:d616721a-e9d2-419e-9d72-a051986bd534",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8d6030c4-2d14-4751-9815-d863e4031fba"
    },
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f",
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:ea1d0942-5ebe-4272-80a4-f5e238416efe",
    "state": "OFFERED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": 

## Consumer accepts the offer (Consumer -> Provider)

In [6]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f",
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:b0553918-b46a-4690-872f-e69cc7fd75fd",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-12T15:55:59.930449Z",
    "updatedAt": "2026-03-12T15:56:00.272429Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:d

## Provider creates the Agreement (Provider -> Consumer)

In [7]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f",
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:ea1d0942-5ebe-4272-80a4-f5e238416efe",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-12T15:55:59.853057Z",
    "updatedAt": "2026-03-12T15:56:00.383241Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:d76f6

## Consumer verifies the agreement (Consumer -> Provider)

In [8]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f",
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:b0553918-b46a-4690-872f-e69cc7fd75fd",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-12T15:55:59.930449Z",
    "updatedAt": "2026-03-12T15:56:00.446151Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:8

## Provider finalizes the negotiation (Provider -> Consumer)

In [9]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:81557c56-19a5-4f8b-b396-641bebfdee5f",
    "providerPid": "urn:provider-pid:d76f62d0-8ae7-49c3-bed2-c619c5d36585",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:ea1d0942-5ebe-4272-80a4-f5e238416efe",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-12T15:55:59.853057Z",
    "updatedAt": "2026-03-12T15:56:00.510760Z",
    "identifiers": {
      "providerPid": "urn:provider-pid

## Final agreement

In [10]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:fde73f7f-64bb-4f46-9e7d-2e99de515a8c",
  "negotiationAgentProcessId": "urn:negotiation-process:ea1d0942-5ebe-4272-80a4-f5e238416efe",
  "negotiationAgentMessageId": "urn:negotiation-message:398aa78c-9f4b-4c7a-ab05-45c819151e10",
  "consumerParticipantId": "did:jwk:consumer",
  "providerParticipantId": "did:jwk:provider",
  "agreementContent": {
    "@id": "urn:agreement:fde73f7f-64bb-4f46-9e7d-2e99de515a8c",
    "@type": "Agreement",
    "assignee": "did:jwk:consumer",
    "assigner": "did:jwk:provider",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "target": "urn:dataset:8d6030c4-2d14-4751-9815-d863e4031fba",
    "timestamp": "1773330960"
  },
  "target": "urn:dataset:8d6030c4-2d14-4751-9815-d863e4031fba",
  "state": "ACTIVE",
  "createdAt": "2026-03-12T15:56:00.389060Z",
  "updatedAt": "2026-03-12T15:56:00.517599Z"
}

Final agreement id: 
urn:agreement:fde73f7f-64bb-4f46-9e7d-2e99de515a8c



# Transfer Negotiation with Agents


## Transfer request setup (Consumer -> Provider)


In [11]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+asd",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["response"]["consumerPid"]
transfer_process_provider_pid = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "agreementId": "urn:agreement:fde73f7f-64bb-4f46-9e7d-2e99de515a8c",
    "format": "http+asd",
    "dataAddress": null,
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "state": "REQUESTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:b1099af8-80cb-45ef-a2ee-415222291ab4",
    "state": "REQUESTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:fde73f7f-64bb-4f46-9e7d-2e99de515a8c",
    "callbackAddress": "http://127.0.0.1:

## Start transfer (Provider -> Consumer)

In [12]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1100/dataplane/proxy/urn:dataplane-transfer:ff60d1f4-96c8-40f6-bebc-fcfdf6da7ce0",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:6265ec34-2ae0-46ba-95f9-c9521bb123e9",
    "state": "STARTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:fde73f7f

## Suspend transfer (Consumer -> Provider)

In [13]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:b1099af8-80cb-45ef-a2ee-415222291ab4",
    "state": "SUSPENDED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:fde73f7f-64bb-4f46-9e7d-2e99de515a8c",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
   

## Restart transfer (Consumer -> Provider)

In [14]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1200/dataplane/proxy/urn:dataplane-transfer:49645cd6-b982-4218-9a60-64221fca45d2",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:b1099af8-80cb-45ef-a2ee-415222291ab4",
    "state": "STARTED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:fde73f7

## Suspension by Provider (Provider -> Consumer)
(Note: Testing provider-initiated suspension)

In [15]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:6265ec34-2ae0-46ba-95f9-c9521bb123e9",
    "state": "SUSPENDED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:fde73f7f-64bb-4f46-9e7d-2e99de515a8c",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
   

## Failure Test: Attempt start with invalid parameters

In [16]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca"
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "HTTP Error 400 Bad Request: {\"@context\":[\"https://w3id.org/dspace/2025/1/context.jsonld\"],\"@type\":\"TransferError\",\"consumerPid\":null,\"providerPid\":null,\"code\":\"6030\",\"reason\":[\"TransferProcessMessageType TransferStartMessage is not allowed here. Current state is SUSPENDED ByProvider\",\"Failed to parse file\"]}"
    ]
  }
}


## Failure Test: Attempt duplicate or invalid suspension

In [17]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "6030",
    "reason": [
      "TransferProcessMessageType TransferSuspensionMessage is not allowed here. Current state is SUSPENDED",
      "Failed to parse file"
    ]
  }
}


## Finalize transfer (Provider -> Consumer)

In [18]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:d12594ab-3fe4-4fa7-92e1-71a53b152e89",
    "providerPid": "urn:provider-pid:b600294d-796d-4715-ba16-74d9e53ce3ca",
    "state": "COMPLETED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:6265ec34-2ae0-46ba-95f9-c9521bb123e9",
    "state": "COMPLETED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:fde73f7f-64bb-4f46-9e7d-2e99de515a8c",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-12T15:56:00.688010Z",